# AEGIS — Data Catalog

**Project:** Adversarial Enforcement and Guardrail Interception System (AEGIS)  
**Purpose:** Comprehensive catalog of all datasets used for training, validation, and evaluation of the AEGIS Layer 1 prompt classifier.  

This notebook documents:
- Data sources, provenance, and licensing
- Schema definitions and label taxonomy
- Per-source sample counts and class contributions
- Data pipeline from raw download to processed splits
- Sample previews from every source
- Quality checks and deduplication statistics

---

## 1 — Label Taxonomy

AEGIS classifies LLM prompts in a healthcare context into **5 mutually exclusive classes**:

| Label | Class Name | Description |
|:-----:|:-----------|:------------|
| 0 | `benign` | Legitimate clinical, biomedical, or administrative queries |
| 1 | `direct_injection` | Explicit prompt-injection attacks that attempt to override system instructions |
| 2 | `indirect_injection` | Malicious payloads hidden inside context the model may process (documents, tool output, etc.) |
| 3 | `jailbreak` | Role-play, persona, or scenario-based attempts to bypass safety restrictions |
| 4 | `phi_extraction` | Attempts to extract Protected Health Information (18 HIPAA identifiers) |

### Design Rationale

- **Direct vs. Indirect Injection:** Separated because the attack surface differs — direct injections target the user prompt field, while indirect injections hide inside trusted context (EHR records, referral letters, tool output). Detection strategies differ accordingly.
- **Jailbreak as a distinct class:** Jailbreak prompts use social engineering (role-play, hypothetical scenarios) rather than technical injection payloads. Their linguistic structure is qualitatively different.
- **PHI Extraction:** Unique to healthcare AI — attempts to coerce the system into disclosing patient data. Aligns with HIPAA's 18 protected identifier categories.

## 2 — Data Source Registry

The table below catalogs every data source used in the AEGIS training pipeline.

| # | Source | Type | Access | Labels Contributed | Reference |
|:-:|:-------|:-----|:-------|:-------------------|:----------|
| 1 | **deepset/prompt-injections** | HuggingFace | Public | `benign`, `direct_injection` | [HF Hub](https://huggingface.co/datasets/deepset/prompt-injections) |
| 2 | **jackhhao/jailbreak-classification** | HuggingFace | Public | `benign`, `jailbreak` | [HF Hub](https://huggingface.co/datasets/jackhhao/jailbreak-classification) |
| 3 | **rubend18/ChatGPT-Jailbreak-Prompts** | HuggingFace | Public | `jailbreak` | [HF Hub](https://huggingface.co/datasets/rubend18/ChatGPT-Jailbreak-Prompts) |
| 4 | **GBaker/MedQA-USMLE-4-options** | HuggingFace | Public | `benign` | Jin et al. (2021), [GitHub](https://github.com/jind11/MedQA) |
| 5 | **PubMedQA** | GitHub clone | Public | `benign` | Jin et al. (2019), [Site](https://pubmedqa.github.io/) |
| 6 | **Tensor Trust** | HuggingFace | Public | `direct_injection` | Toyer et al. (2023), [Site](https://tensortrust.ai/dataset) |
| 7 | **HackAPrompt** | HuggingFace | Public | `direct_injection` | Schulhoff et al. (2023), [HF Hub](https://huggingface.co/datasets/hackaprompt/hackaprompt-dataset) |
| 8 | **Synthetic — Indirect Injection** | Generated | N/A | `indirect_injection` | Template-based, healthcare-contextualized |
| 9 | **Synthetic — PHI Extraction** | Generated | N/A | `phi_extraction` | HIPAA 18-identifier coverage |
| 10 | **Eval Fixtures — Adversarial** | Hand-crafted | In-repo | All attack classes | `test/fixtures/adversarial_corpus.jsonl` |
| 11 | **Eval Fixtures — Benign** | Hand-crafted | In-repo | `benign` | `test/fixtures/benign_corpus.jsonl` |
| 12 | **Eval Fixtures — PHI Scenarios** | Hand-crafted | In-repo | *(L3 evaluation)* | `test/fixtures/phi_scenarios.jsonl` |

In [ ]:
import json
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

CLASS_NAMES = ["benign", "direct_injection", "indirect_injection", "jailbreak", "phi_extraction"]
LABEL_MAP = dict(enumerate(CLASS_NAMES))

DATA_DIR = Path("../../data/processed")
FIXTURES_DIR = Path("../../test/fixtures")


def load_jsonl(path: Path) -> pd.DataFrame:
    records = []
    with open(path) as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

## 3 — Processed Dataset Overview

The preprocessing pipeline (`scripts/data/preprocess.py`) merges all sources into a unified JSONL schema and performs a stratified 80/10/10 split.

In [ ]:
df_train = load_jsonl(DATA_DIR / "train.jsonl")
df_val = load_jsonl(DATA_DIR / "val.jsonl")
df_test = load_jsonl(DATA_DIR / "test.jsonl")
df_all = load_jsonl(DATA_DIR / "all.jsonl")

df_all["label_name"] = df_all["label"].map(LABEL_MAP)

print(f"{'Split':<10} {'Samples':>10}  {'Ratio':>8}")
print("-" * 32)
total = len(df_train) + len(df_val) + len(df_test)
for name, df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    print(f"{name:<10} {len(df):>10,}  {len(df)/total:>8.1%}")
print("-" * 32)
print(f"{'Total':<10} {total:>10,}")

### 3.1 — Unified Record Schema

Every record in the processed JSONL files follows this schema:

```json
{
  "text":   "<prompt text>",
  "label":  0,
  "source": "deepset_injections",
  "split":  "train"
}
```

| Field | Type | Description |
|:------|:-----|:------------|
| `text` | `str` | Raw prompt text (untokenized) |
| `label` | `int` | Class index (0–4), see taxonomy above |
| `source` | `str` | Provenance tag identifying the originating dataset |
| `split` | `str` | One of `train`, `val`, `test` |

In [ ]:
print("Columns:", list(df_all.columns))
print(f"\nData types:\n{df_all.dtypes}")
print(f"\nNull values:\n{df_all.isnull().sum()}")
print(f"\nUnique sources: {df_all['source'].nunique()}")
print(f"Unique labels:  {sorted(df_all['label'].unique())}")

## 4 — Per-Source Breakdown

In [ ]:
source_summary = (
    df_all.groupby("source")
    .agg(
        count=("text", "size"),
        labels=("label_name", lambda x: ", ".join(sorted(x.unique()))),
        avg_chars=("text", lambda x: x.str.len().mean()),
        avg_words=("text", lambda x: x.str.split().str.len().mean()),
    )
    .sort_values("count", ascending=False)
    .reset_index()
)
source_summary["pct"] = (source_summary["count"] / source_summary["count"].sum() * 100).round(1)
source_summary["avg_chars"] = source_summary["avg_chars"].round(0).astype(int)
source_summary["avg_words"] = source_summary["avg_words"].round(1)

source_summary.columns = ["Source", "Count", "Labels", "Avg Chars", "Avg Words", "% of Total"]
source_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Source contribution bar chart
src_counts = df_all["source"].value_counts()
colors = sns.color_palette("tab10", len(src_counts))
src_counts.plot.barh(ax=axes[0], color=colors, edgecolor="black")
axes[0].set_title("Samples per Source", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("")
for i, (v, name) in enumerate(zip(src_counts.values, src_counts.index)):
    axes[0].text(v + 50, i, f"{v:,}", va="center", fontsize=9)

# Source-to-label heatmap
cross = pd.crosstab(df_all["source"], df_all["label_name"])
cross = cross.reindex(columns=CLASS_NAMES, fill_value=0)
cross = cross.loc[src_counts.index]
sns.heatmap(cross, annot=True, fmt=",", cmap="YlOrRd", ax=axes[1], linewidths=0.5, cbar_kws={"shrink": 0.8})
axes[1].set_title("Source × Label Matrix", fontsize=14, fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

## 5 — Per-Label Source Composition

Which sources contribute to each class? This is critical for understanding potential bias — if a class is dominated by a single source, the model may learn source-specific artifacts rather than generalizable features.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5))

for i, cls in enumerate(CLASS_NAMES):
    subset = df_all[df_all["label_name"] == cls]
    src_dist = subset["source"].value_counts()
    if len(src_dist) > 0:
        wedges, texts, autotexts = axes[i].pie(
            src_dist.values,
            labels=None,
            autopct=lambda p: f"{p:.0f}%" if p > 5 else "",
            startangle=90,
            colors=sns.color_palette("Set2", len(src_dist)),
            wedgeprops={"edgecolor": "white", "linewidth": 1},
        )
    axes[i].set_title(f"{cls}\n(n={len(subset):,})", fontsize=11, fontweight="bold")
    axes[i].legend(
        src_dist.index,
        fontsize=7,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.25),
        ncol=1,
    )

plt.suptitle("Source Composition per Class", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 6 — Detailed Source Descriptions & Sample Previews

### 6.1 — deepset/prompt-injections

- **Origin:** HuggingFace Hub (`deepset/prompt-injections`)
- **Description:** Binary prompt-injection detection dataset. Contains both injected and legitimate prompts covering diverse injection techniques.
- **Original labels:** `0` = benign, `1` = injection
- **Mapping:** `1` → `direct_injection`, `0` → `benign`
- **License:** Apache 2.0

In [ ]:
src = df_all[df_all["source"] == "deepset_injections"]
print(f"deepset/prompt-injections: {len(src):,} samples")
print(f"Label distribution: {dict(src['label_name'].value_counts())}")
print(f"\nSample texts:")
for label in src["label_name"].unique():
    print(f"\n  [{label}]")
    for t in src[src["label_name"] == label]["text"].head(2):
        print(f"    → {t[:120]}{'...' if len(t) > 120 else ''}")

### 6.2 — jackhhao/jailbreak-classification

- **Origin:** HuggingFace Hub (`jackhhao/jailbreak-classification`)
- **Description:** Curated dataset of jailbreak attempts and benign prompts for binary classification.
- **Original labels:** `type` field — `"jailbreak"` or `"benign"`
- **Mapping:** `"jailbreak"` → `jailbreak`, `"benign"` → `benign`
- **Text field:** `prompt`

In [ ]:
src = df_all[df_all["source"] == "jailbreak_classification"]
print(f"jailbreak-classification: {len(src):,} samples")
print(f"Label distribution: {dict(src['label_name'].value_counts())}")
print(f"\nSample texts:")
for label in src["label_name"].unique():
    print(f"\n  [{label}]")
    for t in src[src["label_name"] == label]["text"].head(2):
        print(f"    → {t[:120]}{'...' if len(t) > 120 else ''}")

### 6.3 — rubend18/ChatGPT-Jailbreak-Prompts

- **Origin:** HuggingFace Hub (`rubend18/ChatGPT-Jailbreak-Prompts`)
- **Description:** Collection of real ChatGPT jailbreak prompts scraped from public forums.
- **Filtering:** Only prompts with > 20 characters are included.
- **Mapping:** All → `jailbreak`
- **Text field:** `Prompt`

In [ ]:
src = df_all[df_all["source"] == "chatgpt_jailbreak_prompts"]
print(f"ChatGPT-Jailbreak-Prompts: {len(src):,} samples")
print(f"\nSample texts:")
for t in src["text"].head(3):
    print(f"  → {t[:150]}{'...' if len(t) > 150 else ''}")

### 6.4 — GBaker/MedQA-USMLE-4-options

- **Origin:** HuggingFace Hub (`GBaker/MedQA-USMLE-4-options`)
- **Description:** Real USMLE-style medical exam questions. Used to provide domain-relevant benign prompts that test the classifier's ability to distinguish clinical queries from attacks.
- **Mapping:** All → `benign`
- **Text field:** `question`
- **Citation:** Jin et al. "What Disease does this Patient Have? A Large-scale Open Domain Question Answering Dataset from Medical Exams." *Applied Sciences*, 2021.

In [ ]:
src = df_all[df_all["source"] == "medqa"]
print(f"MedQA: {len(src):,} samples (all benign)")
print(f"\nSample texts:")
for t in src["text"].head(3):
    print(f"  → {t[:150]}{'...' if len(t) > 150 else ''}")

### 6.5 — PubMedQA

- **Origin:** GitHub clone (`pubmedqa/pubmedqa`)
- **Description:** Biomedical yes/no/maybe questions derived from PubMed abstracts. Provides another source of domain-specific benign queries for FPR calibration.
- **Mapping:** All → `benign`
- **Text field:** `QUESTION`
- **Citation:** Jin et al. "PubMedQA: A Dataset for Biomedical Research Question Answering." *EMNLP*, 2019.
- **Note:** Requires local clone under `data/raw/pubmedqa/`; skipped if not present.

In [ ]:
src = df_all[df_all["source"] == "pubmedqa"]
if len(src) > 0:
    print(f"PubMedQA: {len(src):,} samples (all benign)")
    print(f"\nSample texts:")
    for t in src["text"].head(3):
        print(f"  → {t[:150]}{'...' if len(t) > 150 else ''}")
else:
    print("PubMedQA: 0 samples (not present in processed data — raw clone required)")

### 6.6 — Tensor Trust

- **Origin:** HuggingFace Hub (`ethz-spylab/tensor-trust-data`)
- **Description:** Large-scale benchmark of prompt injection attacks and defenses from the Tensor Trust game. Includes both hijacking (redirect model output) and extraction (leak system prompt) attack types.
- **Subsets used:**
  - `hijacking_robustness_dataset.jsonl` — output-hijacking attacks
  - `extraction_robustness_dataset.jsonl` — system-prompt extraction attacks
- **Mapping:** All → `direct_injection`
- **Text field:** `attack`
- **Filtering:** Only attacks with > 10 characters
- **Citation:** Toyer et al. "Tensor Trust: Interpretable Prompt Injection Attacks from an Online Game." *ICLR*, 2024.

In [ ]:
for sub in ["tensor_trust_hijacking", "tensor_trust_extraction"]:
    src = df_all[df_all["source"] == sub]
    if len(src) > 0:
        print(f"{sub}: {len(src):,} samples")
        for t in src["text"].head(2):
            print(f"  → {t[:120]}{'...' if len(t) > 120 else ''}")
        print()
    else:
        print(f"{sub}: 0 samples (raw download required)")

### 6.7 — HackAPrompt

- **Origin:** HuggingFace Hub (`hackaprompt/hackaprompt-dataset`)
- **Description:** Successful injection attacks from the HackAPrompt competition. Real adversarial inputs that tricked production LLMs.
- **Mapping:** All → `direct_injection`
- **Text field:** `user_input`
- **Sampling:** Capped at 5,000 samples to maintain class balance (random sample, seed=42).
- **Citation:** Schulhoff et al. "Ignore This Title and HackAPrompt." *EMNLP*, 2023.

In [ ]:
src = df_all[df_all["source"] == "hackaprompt"]
if len(src) > 0:
    print(f"HackAPrompt: {len(src):,} samples (all direct_injection)")
    print(f"\nSample texts:")
    for t in src["text"].head(3):
        print(f"  → {t[:120]}{'...' if len(t) > 120 else ''}")
else:
    print("HackAPrompt: 0 samples (raw download required)")

### 6.8 — Synthetic: Indirect Injection

- **Origin:** Generated by `scripts/data/preprocess.py`
- **Description:** 400 synthetic indirect injection samples. Combines 15 healthcare-context templates (referral letters, EHR output, lab results, etc.) with 20 hidden malicious instruction payloads.
- **Rationale:** No public dataset exists for healthcare-specific indirect injection attacks. These are generated to fill the gap and ensure the classifier can detect hidden payloads embedded in clinical context.
- **Mapping:** All → `indirect_injection`

In [ ]:
src = df_all[df_all["source"] == "synthetic_indirect_injection"]
print(f"Synthetic indirect injection: {len(src):,} samples")
print(f"\nSample texts:")
for t in src["text"].sample(3, random_state=42):
    print(f"  → {t[:150]}{'...' if len(t) > 150 else ''}")
    print()

### 6.9 — Synthetic: PHI Extraction

- **Origin:** Generated by `scripts/data/preprocess.py`
- **Description:** 400 synthetic PHI extraction prompts. Combines 50 PHI-seeking question templates covering all 18 HIPAA identifiers with 13 social-engineering context prefixes.
- **HIPAA Identifiers Covered:** Names, SSNs, dates, addresses, phone/fax numbers, email, MRNs, health plan numbers, account numbers, license numbers, VINs, device IDs, URLs, IPs, biometrics, photographs, other unique identifiers.
- **Mapping:** All → `phi_extraction`

In [ ]:
src = df_all[df_all["source"] == "synthetic_phi_extraction"]
print(f"Synthetic PHI extraction: {len(src):,} samples")
print(f"\nSample texts:")
for t in src["text"].sample(3, random_state=42):
    print(f"  → {t[:150]}{'...' if len(t) > 150 else ''}")

## 7 — Evaluation Fixtures (In-Repo)

These hand-crafted corpora live under `test/fixtures/` and are used for Go integration tests and ablation studies — **not** for training.

In [ ]:
fixture_files = [
    ("adversarial_corpus.jsonl", "Adversarial (L1 eval)"),
    ("benign_corpus.jsonl", "Benign (L1 eval)"),
    ("phi_scenarios.jsonl", "PHI Scenarios (L3 eval)"),
]

for fname, desc in fixture_files:
    path = FIXTURES_DIR / fname
    if path.exists():
        df_fix = load_jsonl(path)
        print(f"{desc} — {fname}")
        print(f"  Records: {len(df_fix)}")
        print(f"  Columns: {list(df_fix.columns)}")
        if "label" in df_fix.columns:
            dist = df_fix["label"].map(LABEL_MAP).value_counts()
            print(f"  Labels:  {dict(dist)}")
        elif "phi_types" in df_fix.columns:
            print(f"  PHI types: {df_fix['phi_types'].explode().value_counts().to_dict()}")
        print(f"  Sample:  {df_fix['text'].iloc[0][:100]}...")
        print()

## 8 — Data Quality Checks

In [ ]:
print("=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)

# Null / empty checks
null_text = df_all["text"].isnull().sum()
empty_text = (df_all["text"].str.strip() == "").sum()
print(f"\nNull text fields:  {null_text}")
print(f"Empty text fields: {empty_text}")

# Duplicate detection
n_dup_exact = df_all["text"].duplicated().sum()
n_dup_lower = df_all["text"].str.lower().str.strip().duplicated().sum()
print(f"\nExact duplicates:      {n_dup_exact:,} ({n_dup_exact/len(df_all)*100:.1f}%)")
print(f"Case-insensitive dups: {n_dup_lower:,} ({n_dup_lower/len(df_all)*100:.1f}%)")

# Duplicates by source
dup_mask = df_all["text"].duplicated(keep=False)
if dup_mask.sum() > 0:
    print(f"\nDuplicate texts by source (texts appearing 2+ times):")
    dup_sources = df_all[dup_mask].groupby("source").size().sort_values(ascending=False)
    for src_name, count in dup_sources.items():
        print(f"  {src_name:<35s} {count:>6,}")

# Label consistency: same text with different labels?
text_labels = df_all.groupby("text")["label"].nunique()
conflicting = text_labels[text_labels > 1]
print(f"\nTexts with conflicting labels: {len(conflicting)}")
if len(conflicting) > 0:
    print("  Example conflicts:")
    for text in conflicting.head(3).index:
        labels = df_all[df_all["text"] == text][["label_name", "source"]].drop_duplicates()
        print(f"    '{text[:80]}...'")
        for _, row in labels.iterrows():
            print(f"      → {row['label_name']} (from {row['source']})")

In [ ]:
# Text length statistics per source
df_all["char_len"] = df_all["text"].str.len()
df_all["word_count"] = df_all["text"].str.split().str.len()

len_stats = (
    df_all.groupby("source")
    .agg(
        n=("text", "size"),
        min_words=("word_count", "min"),
        median_words=("word_count", "median"),
        max_words=("word_count", "max"),
        min_chars=("char_len", "min"),
        median_chars=("char_len", "median"),
        max_chars=("char_len", "max"),
    )
    .sort_values("n", ascending=False)
)

len_stats[["median_words", "median_chars"]] = len_stats[["median_words", "median_chars"]].astype(int)
print("Text length statistics by source:")
len_stats

## 9 — Class Balance & Imbalance Ratio

In [ ]:
class_counts = df_all["label_name"].value_counts().reindex(CLASS_NAMES)
majority = class_counts.max()

balance_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Count": class_counts.values,
    "% of Total": (class_counts.values / class_counts.sum() * 100).round(1),
    "Imbalance Ratio": (majority / class_counts.values).round(2),
})
print("Class balance summary:")
print(balance_df.to_string(index=False))

print(f"\nMost frequent class:  {CLASS_NAMES[class_counts.argmax()]} ({majority:,})")
print(f"Least frequent class: {CLASS_NAMES[class_counts.argmin()]} ({class_counts.min():,})")
print(f"Max imbalance ratio:  {(majority / class_counts.min()):.1f}:1")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = sns.color_palette("muted", len(CLASS_NAMES))

# Stacked bars showing source composition within each class
sources_in_class = df_all.groupby(["label_name", "source"]).size().unstack(fill_value=0)
sources_in_class = sources_in_class.reindex(CLASS_NAMES)
sources_in_class.plot.bar(stacked=True, ax=ax, colormap="Set2", edgecolor="black", linewidth=0.3)

ax.set_title("Class Sizes with Source Breakdown", fontsize=14, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Sample Count")
ax.legend(title="Source", fontsize=7, title_fontsize=9, bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="x", rotation=15)

for i, (cls, count) in enumerate(zip(CLASS_NAMES, class_counts.values)):
    ax.text(i, count + 50, f"{count:,}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## 10 — Data Pipeline Overview

```
┌──────────────────────────────────────────────────────────────────────┐
│                        DATA PIPELINE                                │
├──────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  ┌─────────────────┐    ┌─────────────────┐    ┌────────────────┐   │
│  │  HuggingFace     │    │  GitHub Clones   │    │  Synthetic     │   │
│  │  ─────────────── │    │  ─────────────── │    │  ──────────── │   │
│  │  deepset/        │    │  PubMedQA        │    │  Indirect     │   │
│  │  prompt-inj.     │    │  Tensor Trust    │    │  Injection    │   │
│  │  jailbreak-cls   │    │  HackAPrompt     │    │  (400 samples)│   │
│  │  ChatGPT-JB      │    │                  │    │  PHI Extract. │   │
│  │  MedQA-USMLE     │    │                  │    │  (400 samples)│   │
│  └────────┬─────────┘    └────────┬─────────┘    └───────┬───────┘   │
│           │                       │                      │           │
│           └───────────────────────┼──────────────────────┘           │
│                                   ▼                                  │
│                    ┌──────────────────────────┐                      │
│                    │  scripts/data/preprocess  │                      │
│                    │  ────────────────────────  │                      │
│                    │  • Unified JSONL schema   │                      │
│                    │  • Label mapping (0–4)    │                      │
│                    │  • Text cleaning          │                      │
│                    │  • Dedup / filtering      │                      │
│                    └────────────┬─────────────┘                      │
│                                 ▼                                    │
│                 ┌──────────────────────────────┐                     │
│                 │  Stratified Split (80/10/10) │                     │
│                 └──────┬──────┬──────┬─────────┘                     │
│                        ▼      ▼      ▼                               │
│                   train.jsonl val.jsonl test.jsonl                    │
│                                                                      │
└──────────────────────────────────────────────────────────────────────┘
```

## 11 — Split Verification

In [ ]:
df_train["label_name"] = df_train["label"].map(LABEL_MAP)
df_val["label_name"] = df_val["label"].map(LABEL_MAP)
df_test["label_name"] = df_test["label"].map(LABEL_MAP)

split_dist = pd.DataFrame({
    "Train": df_train["label_name"].value_counts().reindex(CLASS_NAMES),
    "Val": df_val["label_name"].value_counts().reindex(CLASS_NAMES),
    "Test": df_test["label_name"].value_counts().reindex(CLASS_NAMES),
})
split_dist["Total"] = split_dist.sum(axis=1)

print("Samples per class per split:")
print(split_dist)

# Proportions per split
split_pct = split_dist.div(split_dist.sum(axis=0), axis=1).round(3) * 100
print("\nClass proportions (%) per split:")
print(split_pct.round(1))

In [ ]:
# Leakage check
train_texts = set(df_train["text"])
val_texts = set(df_val["text"])
test_texts = set(df_test["text"])

print("Data leakage check (text overlap between splits):")
print(f"  Train ∩ Val  : {len(train_texts & val_texts)}")
print(f"  Train ∩ Test : {len(train_texts & test_texts)}")
print(f"  Val   ∩ Test : {len(val_texts & test_texts)}")

if len(train_texts & val_texts) + len(train_texts & test_texts) + len(val_texts & test_texts) == 0:
    print("  No data leakage detected.")
else:
    print("  WARNING: Overlapping texts found between splits!")

## 12 — References

1. **deepset/prompt-injections** — [HuggingFace](https://huggingface.co/datasets/deepset/prompt-injections). Apache 2.0.

2. **jackhhao/jailbreak-classification** — [HuggingFace](https://huggingface.co/datasets/jackhhao/jailbreak-classification).

3. **rubend18/ChatGPT-Jailbreak-Prompts** — [HuggingFace](https://huggingface.co/datasets/rubend18/ChatGPT-Jailbreak-Prompts).

4. Jin, D., Pan, E., Oufattole, N., Weng, W., Fang, H., & Szolovits, P. (2021). *What Disease does this Patient Have? A Large-scale Open Domain Question Answering Dataset from Medical Exams.* Applied Sciences, 11(14), 6421.

5. Jin, Q., Dhingra, B., Liu, Z., Cohen, W., & Lu, X. (2019). *PubMedQA: A Dataset for Biomedical Research Question Answering.* EMNLP 2019.

6. Toyer, S., Watkins, O., Mendes, E. A., Svegliato, J., Bailey, L., Wang, T., Ong, I., Elmaaroufi, K., Abbeel, P., Darrell, T., Ritter, A., & Russell, S. (2024). *Tensor Trust: Interpretable Prompt Injection Attacks from an Online Game.* ICLR 2024.

7. Schulhoff, S., Pinto, J., Khan, A., Bouchard, L., Si, C., Anati, S., Tagber, V., Kerschbaumer, A., Hestness, J., & Boyd-Graber, J. (2023). *Ignore This Title and HackAPrompt: Exposing Systemic Weaknesses of LLMs through a Global Scale Prompt Hacking Competition.* EMNLP 2023.

8. Dacosta, L. P., Nadeem, M., Joshi, A., & Lashkari, A. H. (2024). *CICIoMT2024: Benchmark dataset for IoMT security.* Internet of Things, 28, 101377.